# Personal photo annotation

Cycle through the cropped photos in `personnal_photos_clean/`, look at each one and record the annotations required for the Texture Passports. Every **Save** writes `personnal_annotations.csv` immediately, and existing annotations are loaded back when you revisit a photo.

Annotate **before** running any of the frozen models on the photos. Pick 3 to 5 DTD terms per photo. Annotator 2 stays empty (single annotator).

[AI-CODE]

In [2]:
# [AI-CODE]
import csv
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "student_files" else HERE / ".."
sys.path.insert(0, str(ROOT.resolve()))
from texturelab.data import DTD_ATTRIBUTES, MATERIALS

STUDENT_DIR = (ROOT / "student_files").resolve()
PHOTO_DIR = STUDENT_DIR / "personnal_photos_clean"
OUT_CSV = ROOT / "student_files" / "personnal_annotations.csv"

# Same columns as manifests/personal_annotations_template.csv, plus free-text notes.
FIELDS = [
    "path", "split", "material", "defect_type", "mask_path", "attributes",
    "annotator_1_terms", "annotator_2_terms", "annotation_completed_before_inference",
    "viewpoint", "scale", "illumination", "exif_removed", "notes",
]

photos = sorted(p for p in PHOTO_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

def rel_path(photo):
    return str(photo.relative_to(STUDENT_DIR))

def load_annotations():
    if not OUT_CSV.exists():
        return {}
    with OUT_CSV.open(newline="", encoding="utf8") as handle:
        return {row["path"]: row for row in csv.DictReader(handle)}

def save_annotations(annotations):
    # Keep rows in photo order so the CSV is stable between saves.
    order = {rel_path(p): i for i, p in enumerate(photos)}
    rows = sorted(annotations.values(), key=lambda r: order.get(r["path"], len(order)))
    with OUT_CSV.open("w", newline="", encoding="utf8") as handle:
        writer = csv.DictWriter(handle, fieldnames=FIELDS)
        writer.writeheader()
        writer.writerows({k: row.get(k, "") for k in FIELDS} for row in rows)

annotations = load_annotations()
print(f"{len(photos)} photos found, {len(annotations)} already annotated")

17 photos found, 17 already annotated


In [3]:
# [AI-CODE]
state = {"index": 0}

image = widgets.Image(format="png", width=400, height=400)
title = widgets.HTML()
status = widgets.HTML()

material = widgets.Dropdown(options=list(MATERIALS) + ["other"], description="Material")
defect = widgets.Text(value="good", description="Defect", placeholder="good, or defect name")
terms = widgets.SelectMultiple(options=DTD_ATTRIBUTES, rows=len(DTD_ATTRIBUTES),
                               description="DTD terms")
extra_terms = widgets.Text(description="Other DTD", placeholder="extra DTD terms; separated by ;")
viewpoint = widgets.Dropdown(options=["frontal", "oblique"], description="Viewpoint")
scale = widgets.Dropdown(options=["close", "medium", "far"], description="Scale")
illumination = widgets.Dropdown(options=["diffuse", "direct", "side-lit", "dim", "mixed"],
                                description="Lighting")
notes = widgets.Textarea(description="Notes", placeholder="defects or factors causing uncertainty",
                         layout=widgets.Layout(width="400px", height="70px"))

prev_button = widgets.Button(description="◀ Previous")
save_button = widgets.Button(description="Save", button_style="success")
next_button = widgets.Button(description="Save + next ▶", button_style="primary")

def selected_terms():
    chosen = list(terms.value)
    chosen += [t.strip() for t in extra_terms.value.split(";") if t.strip()]
    return chosen

def show(index):
    state["index"] = index % len(photos)
    photo = photos[state["index"]]
    with photo.open("rb") as handle:
        image.value = handle.read()
    image.format = photo.suffix.lstrip(".").replace("jpg", "jpeg")
    row = annotations.get(rel_path(photo))
    title.value = (f"<h3>{state['index'] + 1}/{len(photos)} : {photo.name} "
                   f"{'✅' if row else '— not annotated'}</h3>")
    status.value = ""
    row = row or {}
    material.value = row.get("material") or (photo.stem.split("_")[0] if photo.stem.split("_")[0] in MATERIALS else "other")
    defect.value = row.get("defect_type") or "good"
    saved_terms = [t for t in row.get("annotator_1_terms", "").split(";") if t]
    terms.value = tuple(t for t in saved_terms if t in DTD_ATTRIBUTES)
    extra_terms.value = ";".join(t for t in saved_terms if t not in DTD_ATTRIBUTES)
    viewpoint.value = row.get("viewpoint") or viewpoint.options[0]
    scale.value = row.get("scale") or scale.options[0]
    illumination.value = row.get("illumination") or illumination.options[0]
    notes.value = row.get("notes", "")

def save(_=None):
    chosen = selected_terms()
    if not 3 <= len(chosen) <= 5:
        status.value = f"<b style='color:red'>Pick 3 to 5 DTD terms (currently {len(chosen)}). Not saved.</b>"
        return False
    photo = photos[state["index"]]
    annotations[rel_path(photo)] = {
        "path": rel_path(photo),
        "split": "personal",
        "material": material.value,
        "defect_type": defect.value.strip() or "good",
        "mask_path": "",
        # Single annotator, so the consensus attributes are annotator 1's terms.
        "attributes": ";".join(chosen),
        "annotator_1_terms": ";".join(chosen),
        "annotator_2_terms": "",
        "annotation_completed_before_inference": "yes",
        "viewpoint": viewpoint.value,
        "scale": scale.value,
        "illumination": illumination.value,
        "exif_removed": "yes",
        "notes": notes.value.strip(),
    }
    save_annotations(annotations)
    status.value = f"<b style='color:green'>Saved {photo.name} to {OUT_CSV.name}</b>"
    return True

def save_and_next(_):
    if save():
        show(state["index"] + 1)

prev_button.on_click(lambda _: show(state["index"] - 1))
save_button.on_click(save)
next_button.on_click(save_and_next)

form = widgets.VBox([material, defect, terms, extra_terms, viewpoint, scale, illumination, notes])
display(widgets.VBox([
    title,
    widgets.HBox([image, form]),
    widgets.HBox([prev_button, save_button, next_button]),
    status,
]))
show(0)

In [4]:
# Check the saved CSV.
print(OUT_CSV.read_text(encoding="utf8") if OUT_CSV.exists() else "Nothing saved yet")

path,split,material,defect_type,mask_path,attributes,annotator_1_terms,annotator_2_terms,annotation_completed_before_inference,viewpoint,scale,illumination,exif_removed,notes
personnal_photos_clean/grid_chair.jpeg,personal,grid,good,,grid;woven;braided,grid;woven;braided,,yes,frontal,medium,diffuse,yes,"Unusual grid, uneven background lighting"
personnal_photos_clean/grid_round_close.jpeg,personal,grid,good,,grid;porous;pitted,grid;porous;pitted,,yes,frontal,close,diffuse,yes,"Round grid, background variations"
personnal_photos_clean/grid_round_far.jpeg,personal,grid,good,,grid;porous;pitted,grid;porous;pitted,,yes,oblique,far,diffuse,yes,"Round grid, very uneven background"
personnal_photos_clean/grid_square.jpeg,personal,grid,good,,grid;woven;bumpy,grid;woven;bumpy,,yes,frontal,medium,diffuse,yes,Change in background at the borders
personnal_photos_clean/tile_black.jpeg,personal,tile,good,,marbled;blotchy;stained,marbled;blotchy;stained,,yes,frontal,medium,direct,yes,Black tile
perso